**IMPORTS**

In [23]:
print("teste")

teste


In [24]:
import matplotlib.pyplot as plt
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import random_split

In [25]:
batch_size = 64
epochs = 5
lr = 0.001  # learning rate 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


**DOWNLOAD DATASET**

In [26]:
# Download training data from open datasets.
training_data = torch.utils.data.DataLoader(
  torchvision.datasets.MNIST('data', train=True, download=True,
                             transform=torchvision.transforms.Compose([ 
                               torchvision.transforms.ToTensor(),                       
                             ])),
  batch_size=batch_size, shuffle=True)

# Download test data from open datasets.
test_data = torch.utils.data.DataLoader(
  torchvision.datasets.MNIST('data', train=False, download=True,
                             transform=torchvision.transforms.Compose([
                               torchvision.transforms.ToTensor(),
                             ])),
  batch_size=batch_size, shuffle=True)


In [ ]:
figure = plt.figure(figsize=(4, 4))
cols, rows = 3, 3
train_dataset = training_data.dataset

for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(train_dataset), size=(1,)).item()
    img, label = train_dataset[sample_idx]
    ax = figure.add_subplot(rows, cols, i)
    ax.axis("off")
    ax.set_title(str(label))
    ax.imshow(img.squeeze(), cmap="gray")

plt.tight_layout()
plt.show()


In [28]:

# # Create data loaders.
# train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
# test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [ ]:

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(p=0.4)


    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x= F.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x 

model = NeuralNetwork().to(device)
print(model)

In [30]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr= lr)


In [ ]:
def train(dataloader, model, loss_fn, optimizer):
  
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # if batch % 100 == 0:
    loss, current = loss.item(), size
    print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [32]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:

for epoch in range(epochs):
    print(f"Epoch {epoch+1}\n-------------------------------")
    train(training_data, model, loss_fn, optimizer)
    test(test_data, model, loss_fn)
print("Done!")

In [ ]:
model.eval()
images, labels = next(iter(test_data))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    logits = model(images)
    predictions = logits.argmax(dim=1)

figure = plt.figure(figsize=(5, 5))
cols, rows = 3, 3

for i in range(cols * rows):
    ax = figure.add_subplot(rows, cols, i + 1)
    ax.imshow(images[i].cpu().squeeze(), cmap="gray")
    predicted_label = predictions[i].item()
    true_label = labels[i].item()
    ax.set_title(
        f"Pred: {predicted_label} | Real: {true_label}",
        color="green" if predicted_label == true_label else "red",
    )
    ax.axis("off")

plt.tight_layout()
plt.show()